In [2]:
from transformers import pipeline
from datasets import load_dataset
import matplotlib.pyplot as plt
import pandas as pd

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import nltk
from nltk.tokenize import sent_tokenize

from tqdm import tqdm
import torch
nltk.download('punkt')

c:\Users\gdeep\anaconda3\envs\genai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\gdeep\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [4]:
model_id = "google/pegasus-cnn_dailymail"
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [5]:
model = AutoModelForSeq2SeqLM.from_pretrained(model_id).to(device)
dataset = load_dataset("ccdv/arxiv-summarization")

Loading weights: 100%|██████████| 680/680 [00:00<00:00, 8984.63it/s]
[transformers] PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-cnn_dailymail
Key                                  | Status  | 
-------------------------------------+---------+-
model.encoder.embed_positions.weight | MISSING | 
model.decoder.embed_positions.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from datasets import DatasetDict

dataset = DatasetDict({
    "train": dataset["train"].select(range(500)),
    "validation": dataset["validation"].select(range(50)),
    "test": dataset["test"].select(range(50))
}) #not training the full dataset to save time and resources

In [7]:
def createTokens(examples):

    input_encoding = tokenizer(
        examples["article"],
        truncation=True,
        max_length=256 #saving compputation time by limiting the input length to 512 tokens
    )
    target_encoding = tokenizer(
        text_target=examples["abstract"],
        truncation=True,
        max_length=64
    )

    return {
        "input_ids": input_encoding['input_ids'],
        "attention_mask": input_encoding['attention_mask'],
        "labels": target_encoding['input_ids']
    }
    
dataset_tokenized = dataset.map(createTokens,batched=True)

Map: 100%|██████████| 500/500 [00:01<00:00, 330.58 examples/s]


In [8]:
dataset_tokenized

DatasetDict({
    train: Dataset({
        features: ['article', 'abstract', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 5000
    })
    validation: Dataset({
        features: ['article', 'abstract', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 500
    })
    test: Dataset({
        features: ['article', 'abstract', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 500
    })
})

In [9]:
from transformers import DataCollatorForSeq2Seq
sequence_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [10]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    output_dir="./results",

    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,

    fp16=True,
    gradient_checkpointing=True,

    predict_with_generate=True,
    generation_max_length=128,

    eval_strategy="no",
    report_to="none"
)


In [11]:
import evaluate

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    decoded_preds = tokenizer.batch_decode(
        predictions,
        skip_special_tokens=True
    )

    labels = [
        [
            token if token != -100 else tokenizer.pad_token_id
            for token in label
        ]
        for label in labels
    ]

    decoded_labels = tokenizer.batch_decode(
        labels,
        skip_special_tokens=True
    )
    
    decoded_preds = [
        pred.strip() for pred in decoded_preds
    ]

    decoded_labels = [
        label.strip() for label in decoded_labels
    ]

    rouge = evaluate.load("rouge")
    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )

    return {
        key: round(value, 4)
        for key, value in result.items()
    }

In [12]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    processing_class=tokenizer,
    data_collator=sequence_data_collator,
    train_dataset=dataset_tokenized["test"], #doing on test to save time and resources, but should be done on train dataset for real training
    eval_dataset=dataset_tokenized["validation"],
    compute_metrics=compute_metrics
)
model.gradient_checkpointing_enable()
model.config.use_cache = False

trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss


Writing model shards: 100%|██████████| 1/1 [00:05<00:00,  5.51s/it]


TrainOutput(global_step=125, training_loss=16.4713671875, metrics={'train_runtime': 2257.4845, 'train_samples_per_second': 0.221, 'train_steps_per_second': 0.055, 'total_flos': 361183051776000.0, 'train_loss': 16.4713671875, 'epoch': 1.0})

In [ ]:
trainer.state.log_history

[{'train_runtime': 2257.4845,
  'train_samples_per_second': 0.221,
  'train_steps_per_second': 0.055,
  'total_flos': 361183051776000.0,
  'train_loss': 16.4713671875,
  'epoch': 1.0,
  'step': 125}]

In [19]:
small_validation = dataset_tokenized["validation"].select(range(50))

results = trainer.evaluate(
    eval_dataset=small_validation
)

print(results)

KeyboardInterrupt: 

In [20]:
model.save_pretrained("pegasus-finetuned-arxiv")
tokenizer.save_pretrained("pegasus-finetuned-arxiv")

Writing model shards: 100%|██████████| 1/1 [00:08<00:00,  8.98s/it]


('pegasus-finetuned-arxiv\\tokenizer_config.json',
 'pegasus-finetuned-arxiv\\tokenizer.json')

In [22]:
import transformers

print(transformers.__version__)

5.17.0
